# Task 2: EEG Data Visualization
**Goal:** Plot Brainwave Patterns  
**Deliverable:** Charts + Interpretation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as scipy_signal
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
print('Libraries loaded successfully.')

## 1. Simulate EEG Signal

In [ ]:
np.random.seed(7)

# Sampling parameters
fs = 256          # Sampling frequency (Hz) — standard EEG
duration = 10     # seconds
t = np.linspace(0, duration, int(fs * duration))

# Brainwave bands
bands = {
    'Delta (0.5–4 Hz)':   {'freq': 2.0,  'amp': 50.0, 'color': '#E91E63'},
    'Theta (4–8 Hz)':     {'freq': 6.0,  'amp': 30.0, 'color': '#9C27B0'},
    'Alpha (8–13 Hz)':    {'freq': 10.5, 'amp': 40.0, 'color': '#2196F3'},
    'Beta (13–30 Hz)':    {'freq': 20.0, 'amp': 20.0, 'color': '#4CAF50'},
    'Gamma (30–100 Hz)':  {'freq': 45.0, 'amp': 10.0, 'color': '#FF9800'}
}

# Generate individual band signals
band_signals = {}
for name, params in bands.items():
    noise = np.random.normal(0, params['amp'] * 0.15, len(t))
    band_signals[name] = params['amp'] * np.sin(2 * np.pi * params['freq'] * t) + noise

# Composite EEG signal
eeg_raw = sum(band_signals.values()) + np.random.normal(0, 5, len(t))

print(f'EEG signal: {len(t)} samples | Duration: {duration}s | Fs: {fs} Hz')
print(f'Amplitude range: [{eeg_raw.min():.1f}, {eeg_raw.max():.1f}] µV')

## 2. Brainwave Pattern Plots

In [ ]:
fig = plt.figure(figsize=(18, 20))
fig.suptitle('EEG Brainwave Pattern Visualization', fontsize=18, fontweight='bold', y=0.99)
gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.55, wspace=0.3)

# --- Raw EEG Signal (full width) ---
ax_raw = fig.add_subplot(gs[0, :])
ax_raw.plot(t[:fs*4], eeg_raw[:fs*4], color='#263238', linewidth=0.7, alpha=0.9)
ax_raw.set_xlabel('Time (s)')
ax_raw.set_ylabel('Amplitude (µV)')
ax_raw.set_title('Raw Composite EEG Signal (first 4 seconds)', fontweight='bold', fontsize=12)
ax_raw.fill_between(t[:fs*4], eeg_raw[:fs*4], alpha=0.1, color='#2196F3')

# --- Individual Brainwave Bands ---
positions = [(1,0),(1,1),(2,0),(2,1),(3,0)]
for idx, (name, params) in enumerate(bands.items()):
    r, c = positions[idx]
    ax = fig.add_subplot(gs[r, c])
    seg = band_signals[name][:fs*4]
    ax.plot(t[:fs*4], seg, color=params['color'], linewidth=0.9)
    ax.set_title(name, fontweight='bold', color=params['color'])
    ax.set_xlabel('Time (s)', fontsize=9)
    ax.set_ylabel('µV', fontsize=9)
    ax.tick_params(labelsize=8)

# --- Power Spectral Density ---
ax_psd = fig.add_subplot(gs[3, 1])
freqs, psd = scipy_signal.welch(eeg_raw, fs=fs, nperseg=512)
ax_psd.semilogy(freqs[:200], psd[:200], color='#37474F', linewidth=1.2)
band_ranges = [(0.5,4,'#E91E63','Delta'),(4,8,'#9C27B0','Theta'),
               (8,13,'#2196F3','Alpha'),(13,30,'#4CAF50','Beta'),(30,100,'#FF9800','Gamma')]
for lo, hi, col, lbl in band_ranges:
    mask = (freqs >= lo) & (freqs <= hi)
    ax_psd.fill_between(freqs[mask], psd[mask], alpha=0.35, color=col, label=lbl)
ax_psd.set_xlim(0, 60)
ax_psd.set_xlabel('Frequency (Hz)', fontsize=9)
ax_psd.set_ylabel('PSD (µV²/Hz)', fontsize=9)
ax_psd.set_title('Power Spectral Density', fontweight='bold')
ax_psd.legend(fontsize=7, loc='upper right')

# --- Band Power Bar Chart ---
ax_bar = fig.add_subplot(gs[4, :])
band_powers = {}
for lo, hi, col, lbl in band_ranges:
    mask = (freqs >= lo) & (freqs <= hi)
    band_powers[lbl] = np.trapz(psd[mask], freqs[mask])
colors_bar = ['#E91E63','#9C27B0','#2196F3','#4CAF50','#FF9800']
bp_vals = list(band_powers.values())
bp_labels = list(band_powers.keys())
bars = ax_bar.bar(bp_labels, bp_vals, color=colors_bar, edgecolor='white', linewidth=0.8)
ax_bar.bar_label(bars, fmt='%.1f', padding=3, fontsize=10)
ax_bar.set_ylabel('Band Power (µV²)', fontsize=10)
ax_bar.set_title('Relative Band Power Comparison', fontweight='bold', fontsize=12)

plt.savefig('eeg_brainwave_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('EEG charts saved as eeg_brainwave_charts.png')

## 3. Interpretation

In [ ]:
dominant = max(band_powers, key=band_powers.get)
print('=== EEG BRAINWAVE INTERPRETATION ===')
print(f'\nDominant Band: {dominant} ({band_powers[dominant]:.1f} µV²)')
print('\nBand Power Summary:')
total_power = sum(band_powers.values())
for band, power in sorted(band_powers.items(), key=lambda x: -x[1]):
    pct = power / total_power * 100
    print(f'  {band:<8}: {power:8.2f} µV²  ({pct:.1f}%)')

print('\n=== Clinical Interpretation ===')
interpretations = {
    'Delta': 'Deep sleep, unconsciousness. High delta in awake state may indicate pathology.',
    'Theta': 'Light sleep, drowsiness, meditation. Linked to memory encoding.',
    'Alpha': 'Relaxed wakefulness, eyes closed. Indicates calm, focused state.',
    'Beta': 'Active thinking, focus, problem-solving. Normal waking consciousness.',
    'Gamma': 'High-level cognitive processing, perception binding. Associated with learning.'
}
for band, desc in interpretations.items():
    print(f'\n{band}: {desc}')